 # Calculate operator punctuality stats based on Chuuchuu data
 
 This notebook aims to used the merge enriched parquet, to calculate specific outputs about operator's punctuality.

 The overall aim is to compare what we can calculate with Chuuchuu, with operator official statistics (when available)

In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_columns", None)   

In [2]:
from chuuchuu_paths import output_path

PARQUET_PATH = output_path("2025_01_to_2025_12")

OPERATORS_OF_INTEREST = ["DB", "Eurostar", "Flixtrain", "NS", "OEBB", "Ouigo", "SBB",
                         "SNCB", "SNCF", "Trenitalia", "Trenitalia France", "Westbahn"]

# columns with few distinct values: loaded as pandas categories (much lighter than one string per row)
CATEGORY_COLUMNS = [
    "date", "agency", "routeType", "journey_type", "country","HSR_service", "LD", "night_train", "joint_service",
    "operator_for_punctuality", "normalized_operator", "depart_terminus", "arrivalCancelled_resolved",
    "stopName",  # needed for SNCB's Brussels stops
]
OTHER_COLUMNS = ["journey_id", "journey_id_country", "is_ambiguous_trip", "sort_time", "arrivalDelay"]

# subset at read time: only the columns of interest, only the stops of the operators of interest,
# without the stops whose journey_type couldn't be resolved ("unknown"),
# so the full 46-column parquet is never loaded in memory
data_chuuchuu = pq.read_table(
    PARQUET_PATH,
    columns=CATEGORY_COLUMNS + OTHER_COLUMNS,
    filters=[
        ("operator_for_punctuality", "in", OPERATORS_OF_INTEREST),
    ],
    read_dictionary=CATEGORY_COLUMNS,
).to_pandas()

# drop the categories no row uses anymore (e.g. the 15 other operators, "unknown" journey_type)
data_chuuchuu[CATEGORY_COLUMNS] = data_chuuchuu[CATEGORY_COLUMNS].apply(lambda col: col.cat.remove_unused_categories())

print(f"{len(data_chuuchuu)} stops, {data_chuuchuu.memory_usage(deep=True).sum() / 1e9:.1f} GB in memory")
print(data_chuuchuu["operator_for_punctuality"].value_counts())
print(data_chuuchuu["journey_type"].value_counts())

78860075 stops, 7.8 GB in memory
operator_for_punctuality
Trenitalia           33595498
NS                   18299425
SNCB                 12524649
SNCF                  6395258
DB                    3303197
SBB                   3049674
OEBB                  1121412
Westbahn               229988
Eurostar               143836
Ouigo                  112579
Flixtrain               58321
Trenitalia France       26238
Name: count, dtype: int64
journey_type
domestic         74540758
international     4314107
unknown              5210
Name: count, dtype: int64


# Adding columns for punctuality

In [3]:
# delayed by strictly more than X min at this stop (arrivalDelay is in seconds): a stop exactly
# X:00 late is not delayed at that threshold. A cancelled stop, or one with no measured delay
# (NaN), is never counted as delayed.
DELAY_THRESHOLDS_MIN = [3, 5, 6, 10, 15, 30, 60]

not_cancelled = data_chuuchuu["arrivalCancelled_resolved"] != "t"
for threshold in DELAY_THRESHOLDS_MIN:
    data_chuuchuu[f"delayed_{threshold}min"] = not_cancelled & (data_chuuchuu["arrivalDelay"] > threshold * 60)

print(data_chuuchuu[[f"delayed_{t}min" for t in DELAY_THRESHOLDS_MIN]].sum())

delayed_3min     13152838
delayed_5min      7944032
delayed_6min      6524046
delayed_10min     3499491
delayed_15min     2108430
delayed_30min      793160
delayed_60min      210984
dtype: int64


# DB

DB is using the following parameter to calculate punctuality :

* Services: Long distance
* Threshold (min): 6
* Measurement: at all stops
* Train or passenger punctuality: train punctuality (<i>pax pctlty also available</i>)
* Account for cancelled trains: No
* Account joint services: cover all collaborations on the German network

NB: for DB, we calculate at the stop level, to be able to compare with their own stat: we can keep rows where we are not sure the stop is terminus or depart. For othter operators, do not forget to do the filter

In [17]:
display(data_chuuchuu[(data_chuuchuu["operator_for_punctuality"]=="DB")&(data_chuuchuu["arrivalCancelled_resolved"]=="f")])


,date,agency,routeType,journey_type,country,HSR_service,LD,night_train,joint_service,operator_for_punctuality,normalized_operator,depart_terminus,arrivalCancelled_resolved,stopName,journey_id,journey_id_country,is_ambiguous_trip,sort_time,arrivalDelay,delayed_3min,delayed_5min,delayed_6min,delayed_10min,delayed_15min,delayed_30min,delayed_60min
1361,2025-01-01,DB,ICE,domestic,Germany,Yes,Yes,No,DB,DB,DB,depart,f,München Hbf,DB_ICE_1000_2025-01-01,DB_ICE_1000_2025-01-01_Germany,False,2025-01-01 16:51:00+00:00,NaN,False,False,False,False,False,False,False
1362,2025-01-01,DB,ICE,domestic,Germany,Yes,Yes,No,DB,DB,DB,intermediate,f,Nürnberg Hbf,DB_ICE_1000_2025-01-01,DB_ICE_1000_2025-01-01_Germany,False,2025-01-01 17:56:00+00:00,0.0,False,False,False,False,False,False,False
1363,2025-01-01,DB,ICE,domestic,Germany,Yes,Yes,No,DB,DB,DB,intermediate,f,Erfurt Hbf,DB_ICE_1000_2025-01-01,DB_ICE_1000_2025-01-01_Germany,False,2025-01-01 19:09:00+00:00,60.0,False,False,False,False,False,False,False
1364,2025-01-01,DB,ICE,domestic,Germany,Yes,Yes,No,DB,DB,DB,intermediate,f,Halle(Saale)Hbf,DB_ICE_1000_2025-01-01,DB_ICE_1000_2025-01-01_Germany,False,2025-01-01 19:38:00+00:00,120.0,False,False,False,False,False,False,False
1365,2025-01-01,DB,ICE,domestic,Germany,Yes,Yes,No,DB,DB,DB,intermediate,f,Berlin Südkreuz,DB_ICE_1000_2025-01-01,DB_ICE_1000_2025-01-01_Germany,False,2025-01-01 20:46:00+00:00,240.0,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78860069,2025-12-31,DB,ECE,international,Germany,No,Yes,No,DB_SBB,DB,DB,intermediate,f,Frankfurt(Main)Hbf,dedup_ECE_9_2025-12-31,dedup_ECE_9_2025-12-31_Germany,False,2025-12-31 19:14:00+00:00,120.0,False,False,False,False,False,False,False
78860070,2025-12-31,DB,ECE,international,Germany,No,Yes,No,DB_SBB,DB,DB,intermediate,f,Mannheim Hbf,dedup_ECE_9_2025-12-31,dedup_ECE_9_2025-12-31_Germany,False,2025-12-31 19:57:00+00:00,420.0,True,True,True,False,False,False,False
78860071,2025-12-31,DB,ECE,international,Germany,No,Yes,No,DB_SBB,DB,DB,intermediate,f,Karlsruhe Hbf,dedup_ECE_9_2025-12-31,dedup_ECE_9_2025-12-31_Germany,False,2025-12-31 20:23:00+00:00,180.0,False,False,False,False,False,False,False
78860072,2025-12-31,DB,ECE,international,Germany,No,Yes,No,DB_SBB,DB,DB,intermediate,f,Offenburg,dedup_ECE_9_2025-12-31,dedup_ECE_9_2025-12-31_Germany,False,2025-12-31 20:58:00+00:00,0.0,False,False,False,False,False,False,False


In [18]:
display(data_chuuchuu["LD"][(data_chuuchuu["operator_for_punctuality"]=="DB")&(data_chuuchuu["arrivalCancelled_resolved"]=="f")].unique())


['Yes', 'No']
Categories (2, str): ['No', 'Yes']

In [4]:
# long distance trains only, joint services included (every stop on the German network is DB's)
db = data_chuuchuu[(data_chuuchuu["operator_for_punctuality"] == "DB") & (data_chuuchuu["LD"] == "Yes")]

# STOP LEVEL (DB's own method): every stop with an arrival delay measured, cancellations not counted
db_measured = db[(db["arrivalCancelled_resolved"] == "f") & db["arrivalDelay"].notna()]
DB_5MIN_S = 5 * 60   # delayed when arrivalDelay > 300 s (same as delayed_5min)

# TERMINUS LEVEL: one row per punctuality unit (journey_id_country = the train on the German network),
# its closing stop = the terminus, or for a train that leaves Germany the last stop before the border.
# A departure-only segment has no arrival to measure and is left out
db_ranked = db[(db["depart_terminus"] != "unknown") & db["sort_time"].notna()]
db_terminus = db_ranked.sort_values("sort_time", kind="stable").groupby("journey_id_country", observed=True).tail(1)
db_terminus = db_terminus[db_terminus["depart_terminus"] != "depart"]

# cancellation status read on the unit's stops: partial = at least one stop cancelled, full = every stop
share_stops_cancelled = (db["arrivalCancelled_resolved"] == "t").groupby(db["journey_id_country"], observed=True).mean()
cancelled_units = set(share_stops_cancelled.index[share_stops_cancelled > 0])  # full + partial
fully_cancelled_units = set(share_stops_cancelled.index[share_stops_cancelled == 1])

db_terminus = db_terminus.assign(
    train_cancelled=db_terminus["journey_id_country"].isin(cancelled_units),
    train_fully_cancelled=db_terminus["journey_id_country"].isin(fully_cancelled_units),
    terminus_cancelled=db_terminus["arrivalCancelled_resolved"] == "t",
)
# a partially cancelled train counts as not on time only if it does not reach its terminus
db_terminus["not_reaching_terminus"] = db_terminus["train_fully_cancelled"] | (
    db_terminus["train_cancelled"] & db_terminus["terminus_cancelled"]
)

# served terminus with a measured delay (cancellations not counted)
db_terminus_served = db_terminus[~db_terminus["terminus_cancelled"] & db_terminus["arrivalDelay"].notna()]
# kept when it can be judged: the train does not reach its terminus, or the delay is measured there
t = db_terminus[db_terminus["not_reaching_terminus"] | db_terminus["arrivalDelay"].notna()]
t_late = ((t["arrivalDelay"] > DB_5MIN_S) & ~t["terminus_cancelled"]) | t["not_reaching_terminus"]

# day trains with a resolved journey_type only (last row)
keep = (t["night_train"] != "Yes") & (t["journey_type"] != "unknown")

# (threshold, measured_at, cancellation rule, rows measured, late mask)
DB_CASES = [
    ("6 min", "all stops", "not counted", db_measured, db_measured["delayed_6min"]),
    ("5 min", "all stops", "not counted", db_measured, db_measured["arrivalDelay"] > DB_5MIN_S),
    ("5 min", "terminus / last stop before border", "not counted", db_terminus_served,
        db_terminus_served["arrivalDelay"] > DB_5MIN_S),
    ("5 min", "terminus / last stop before border",
        "fully cancelled + partially cancelled not reaching terminus", t, t_late),
    ("5 min", "terminus / last stop before border, no night trains, journey_type known",
        "fully cancelled + partially cancelled not reaching terminus", t[keep], t_late[keep]),
]

db_results = pd.DataFrame([{
    "services": "DB long distance",
    "threshold": threshold,
    "measured_at": measured_at,
    "cancellation_counted_as_delayed": rule,
    "n_measured": len(rows),
    "n_delayed": late.sum(),  # delayed (+ not-on-time cancelled trains where counted)
    "punctuality_chuuchuu_%": round((1 - late.mean()) * 100, 1),
} for threshold, measured_at, rule, rows, late in DB_CASES])

display(db_results)

,services,threshold,measured_at,cancellation_counted_as_delayed,n_measured,n_delayed,punctuality_chuuchuu_%
0,DB long distance,6 min,all stops,not counted,2875612,1182335,58.9
1,DB long distance,5 min,all stops,not counted,2875612,1288378,55.2
2,DB long distance,5 min,terminus / last stop before border,not counted,313701,164492,47.6
3,DB long distance,5 min,terminus / last stop before border,fully cancelled + partially cancelled not reac...,332785,183576,44.8
4,DB long distance,5 min,"terminus / last stop before border, no night t...",fully cancelled + partially cancelled not reac...,322961,178496,44.7


# Eurostar

Eurostar is likely using the following parameters to calculate punctuality:

* Services: geographical groups of routes (Routes Belgium, London routes, Continental routes)
* Threshold (min): 6 (Belgium), 5 and 15 (London), 15 (Continental)
* Measurement: likely at terminus
* Train or passenger punctuality: train punctuality
* Account for cancelled trains: tested here as yes for London and Continental routes (a train with at least one cancelled stop, fully or partially cancelled, counts as delayed), no for Belgium
* Account joint services: to ask

How the route groups are built here:

* **Routes Belgium**: measured at stop level, on every Belgian station Eurostar calls at (arrivals only : not cancelled, delay measured)
* **London routes** (measured at terminus, one row per train, cancelled trains counted as delayed): London St Pancras is Eurostar's only UK station in the data, so a train touches London when it has a stop in the United Kingdom. Two versions, since we don't know which one Eurostar uses:
  * *opt 1, only arriving St Pancras*: the terminus is St Pancras
  * *opt 2, arriving + departing St Pancras*: St Pancras is the terminus or the departure (a train leaving London is measured at its continental terminus)
* **Continental routes** (measured at terminus, cancelled trains counted as delayed): trains with no stop in the United Kingdom
* **Continental routes, stop level** (measured on every arrival station, not only the terminus):
  * *all stops of non-London routes*: every station of the trains with no stop in the United Kingdom
  * *all stops except St Pancras*: every station of every train (UK routes included), St Pancras excluded
  * cancellation, two versions: *full + partial* = every cancelled stop counts as delayed; *full cancellation only* = only the stops of fully cancelled trains count as delayed, the cancelled stops of partially cancelled trains are left out

In [20]:
eurostar = data_chuuchuu[data_chuuchuu["operator_for_punctuality"] == "Eurostar"]

# journeys touching London (St Pancras = Eurostar's only UK station)
london_journeys = set(eurostar.loc[eurostar["country"] == "United Kingdom", "journey_id"])

# train cancellation: partial = at least one stop cancelled, full = every stop cancelled
share_stops_cancelled = (eurostar["arrivalCancelled_resolved"] == "t").groupby(eurostar["journey_id"]).mean()
cancelled_journeys = set(share_stops_cancelled.index[share_stops_cancelled > 0])
fully_cancelled_journeys = set(share_stops_cancelled.index[share_stops_cancelled == 1])

# STOP LEVEL: every arrival station (a train's departure station has no arrival, so it's left out),
# kept when it can be judged: cancelled, or arrival delay measured. is_cancelled = this stop is cancelled
eurostar_stops = eurostar[eurostar["depart_terminus"] != "depart"]
eurostar_stops = eurostar_stops.assign(
    is_cancelled=eurostar_stops["arrivalCancelled_resolved"] == "t",
    train_fully_cancelled=eurostar_stops["journey_id"].isin(fully_cancelled_journeys),
)
eurostar_stops = eurostar_stops[eurostar_stops["is_cancelled"] | eurostar_stops["arrivalDelay"].notna()]

eurostar_stops_served = eurostar_stops[~eurostar_stops["is_cancelled"]]  # cancellations not counted
# only full cancellation counted: the cancelled stops of partially cancelled trains are left out
eurostar_stops_full_only = eurostar_stops[~eurostar_stops["is_cancelled"] | eurostar_stops["train_fully_cancelled"]]

# TRAIN LEVEL: one row per train, its terminus. Kept when it can be judged: cancelled, or arrival
# delay measured at the terminus. is_cancelled = the train is fully or partially cancelled
eurostar_terminus = eurostar[eurostar["depart_terminus"] == "terminus"]
eurostar_terminus = eurostar_terminus.assign(
    is_cancelled=eurostar_terminus["journey_id"].isin(cancelled_journeys),
    train_fully_cancelled=eurostar_terminus["journey_id"].isin(fully_cancelled_journeys),
)
eurostar_terminus = eurostar_terminus[eurostar_terminus["is_cancelled"] | eurostar_terminus["arrivalDelay"].notna()]

is_london_stop = eurostar_stops["journey_id"].isin(london_journeys)
is_london_stop_full_only = eurostar_stops_full_only["journey_id"].isin(london_journeys)

# route group -> (where it is measured, how cancellations are counted, rows measured);
# a row counts as late when delayed by more than the threshold OR is_cancelled
eurostar_routes = {
    "Routes Belgium":
        ("all BE stops", "not counted", eurostar_stops_served[eurostar_stops_served["country"] == "Belgium"]),
    "London routes - opt 1 (only arriving St Pancras)":
        ("terminus", "train cancelled (full + partial)", eurostar_terminus[eurostar_terminus["country"] == "United Kingdom"]),
    "London routes - opt 2 (arriving + departing St Pancras)":
        ("terminus", "train cancelled (full + partial)", eurostar_terminus[eurostar_terminus["journey_id"].isin(london_journeys)]),
    "Continental routes":
        ("terminus", "train cancelled (full + partial)", eurostar_terminus[~eurostar_terminus["journey_id"].isin(london_journeys)]),
    "Continental routes - all stops of non-London routes":
        ("all stops", "stop cancelled (full + partial)", eurostar_stops[~is_london_stop]),
    "Continental routes - all stops except St Pancras":
        ("all stops except St Pancras", "stop cancelled (full + partial)", eurostar_stops[eurostar_stops["country"] != "United Kingdom"]),
    "Continental routes - all stops of non-London routes (full cancellation only)":
        ("all stops", "stop of a fully cancelled train", eurostar_stops_full_only[~is_london_stop_full_only]),
    "Continental routes - all stops except St Pancras (full cancellation only)":
        ("all stops except St Pancras", "stop of a fully cancelled train", eurostar_stops_full_only[eurostar_stops_full_only["country"] != "United Kingdom"]),
}

# (route group, threshold in min)
EUROSTAR_CASES = [
    ("Routes Belgium", 6),
    ("London routes - opt 1 (only arriving St Pancras)", 5),
    ("London routes - opt 2 (arriving + departing St Pancras)", 5),
    ("London routes - opt 1 (only arriving St Pancras)", 15),
    ("London routes - opt 2 (arriving + departing St Pancras)", 15),
    ("Continental routes", 15),
    ("Continental routes - all stops of non-London routes", 15),
    ("Continental routes - all stops except St Pancras", 15),
    ("Continental routes - all stops of non-London routes (full cancellation only)", 15),
    ("Continental routes - all stops except St Pancras (full cancellation only)", 15),
]

eurostar_results = []
for route, threshold in EUROSTAR_CASES:
    measured_at, cancellation_rule, rows = eurostar_routes[route]
    late = rows[f"delayed_{threshold}min"] | rows["is_cancelled"]
    eurostar_results.append({
        "services": route,
        "threshold_min": threshold,
        "measured_at": measured_at,
        "cancellation_counted_as_delayed": cancellation_rule,
        "n_measured": len(rows),  # stops for stop-level groups, trains for "terminus"
        # cancelled rows, split by whether their train is partially or fully cancelled
        # (trains for "terminus", stops for stop-level groups; 0 where cancellations are left out)
        "n_cancelled_partial": (rows["is_cancelled"] & ~rows["train_fully_cancelled"]).sum(),
        "n_cancelled_full": (rows["is_cancelled"] & rows["train_fully_cancelled"]).sum(),
        "n_cancelled_full_or_partial": rows["is_cancelled"].sum(),
        "n_delayed": late.sum(),  # delayed + cancelled
        "punctuality_chuuchuu_%": round((1 - late.mean()) * 100, 1),
    })
eurostar_results = pd.DataFrame(eurostar_results)

display(eurostar_results)

,services,threshold_min,measured_at,cancellation_counted_as_delayed,n_measured,n_cancelled_partial,n_cancelled_full,n_cancelled_full_or_partial,n_delayed,punctuality_chuuchuu_%
0,Routes Belgium,6,all BE stops,not counted,32037,0,0,0,7164,77.6
1,London routes - opt 1 (only arriving St Pancras),5,terminus,train cancelled (full + partial),9162,210,211,421,2356,74.3
2,London routes - opt 2 (arriving + departing St...,5,terminus,train cancelled (full + partial),18270,326,393,719,4086,77.6
3,London routes - opt 1 (only arriving St Pancras),15,terminus,train cancelled (full + partial),9162,210,211,421,1626,82.3
4,London routes - opt 2 (arriving + departing St...,15,terminus,train cancelled (full + partial),18270,326,393,719,2773,84.8
5,Continental routes,15,terminus,train cancelled (full + partial),22198,2142,1233,3375,6744,69.6
6,Continental routes - all stops of non-London r...,15,all stops,stop cancelled (full + partial),74473,4252,2401,6653,17882,76.0
7,Continental routes - all stops except St Pancras,15,all stops except St Pancras,stop cancelled (full + partial),94093,4628,2692,7320,20444,78.3
8,Continental routes - all stops of non-London r...,15,all stops,stop of a fully cancelled train,70221,0,2401,2401,13630,80.6
9,Continental routes - all stops except St Pancr...,15,all stops except St Pancras,stop of a fully cancelled train,89465,0,2692,2692,15816,82.3


<!--
## Punctuality units

Same logic as Step 7 of `Chuuchuu_pipeline.ipynb`, run on the full year: stops are collapsed into punctuality units (one per national segment for incumbents, one per whole journey for `operator_exception` operators), then each unit gets its cancellation status and its arrival outcome at the closing stop (terminus, or last stop before the border).
-->

# Flixtrain

In [21]:
# No official data for 2025 to compare to

# NS

The issue with NS is that their statistics is at <i><b>passenger level</i></b>

In 2025 they use 3 and 10 min threshold in their report
Punctuality is measured at 47 different points on the network, and his display for their whole services. Cancelled trains are considered as not on time



In [22]:
ns = data_chuuchuu[data_chuuchuu["operator_for_punctuality"] == "NS"]

ns_stops = ns[ns["depart_terminus"] != "depart"]
ns_stops = ns_stops.assign(is_cancelled=ns_stops["arrivalCancelled_resolved"] == "t")
ns_stops = ns_stops[ns_stops["is_cancelled"] | ns_stops["arrivalDelay"].notna()]

# a stop counts as late when delayed by more than the threshold OR cancelled at this stop
NS_THRESHOLDS_MIN = [3, 10]

ns_results = []
for threshold in NS_THRESHOLDS_MIN:
    late = ns_stops[f"delayed_{threshold}min"] | ns_stops["is_cancelled"]
    ns_results.append({
        "services": "all NS services",
        "threshold_min": threshold,
        "measured_at": "all stops",
        "cancellation_counted_as_delayed": "stop cancelled (full + partial)",
        "n_measured": len(ns_stops),
        "n_cancelled": ns_stops["is_cancelled"].sum(),
        "n_delayed": late.sum(),  # delayed + cancelled
        "punctuality_chuuchuu_%": round((1 - late.mean()) * 100, 1),
    })
ns_results = pd.DataFrame(ns_results)

display(ns_results)

,services,threshold_min,measured_at,cancellation_counted_as_delayed,n_measured,n_cancelled,n_delayed,punctuality_chuuchuu_%
0,all NS services,3,all stops,stop cancelled (full + partial),16432212,653421,2040628,87.6
1,all NS services,10,all stops,stop cancelled (full + partial),16432212,653421,861808,94.8


# ÖBB

Info from ÖBB on their punctuality metrics:

1. Threshold: A passenger train counts as punctual if it arrives with a delay of no more than 5 minutes and 29 seconds. Punctuality is measured at all scheduled arrivals at stations in Austria, not only at the terminus.

2. Cancelled trains: Cancellations are not included in our punctuality statistics.

3. Jointly operated services: The statistics cover all trains operated by ÖBB-Personenverkehr AG within ÖBBs rail network, including services run in cooperation with partner railways.

In [23]:
data_chuuchuu[data_chuuchuu["operator_for_punctuality"]=="OEBB"].head(100)

,date,agency,routeType,journey_type,country,HSR_service,LD,night_train,joint_service,operator_for_punctuality,normalized_operator,depart_terminus,arrivalCancelled_resolved,stopName,journey_id,journey_id_country,is_ambiguous_trip,sort_time,arrivalDelay,delayed_3min,delayed_5min,delayed_6min,delayed_10min,delayed_15min,delayed_30min,delayed_60min
23995,2025-01-01,OEBB,EC,international,Austria,No,Yes,No,Ceske Drahy_OEBB_PKP,OEBB,OEBB,depart,f,Wien Hbf,OEBB_EC_100_2025-01-01,OEBB_EC_100_2025-01-01_Austria,False,2025-01-01 17:10:00+00:00,NaN,False,False,False,False,False,False,False
23996,2025-01-01,OEBB,EC,international,Austria,No,Yes,No,Ceske Drahy_OEBB_PKP,OEBB,OEBB,last stop crossborder,f,Bernhardsthal,OEBB_EC_100_2025-01-01,OEBB_EC_100_2025-01-01_Austria,False,2025-01-01 17:59:00+00:00,540.0,True,True,True,False,False,False,False
24026,2025-01-02,OEBB,EC,international,Austria,No,Yes,No,Ceske Drahy_OEBB_PKP,OEBB,OEBB,depart,f,Wien Hbf,OEBB_EC_100_2025-01-02,OEBB_EC_100_2025-01-02_Austria,False,2025-01-02 17:10:00+00:00,NaN,False,False,False,False,False,False,False
24052,2025-01-03,OEBB,EC,international,Austria,No,Yes,No,Ceske Drahy_OEBB_PKP,OEBB,OEBB,depart,f,Wien Hbf,OEBB_EC_100_2025-01-03,OEBB_EC_100_2025-01-03_Austria,False,2025-01-03 17:10:00+00:00,NaN,False,False,False,False,False,False,False
24079,2025-01-04,OEBB,EC,international,Austria,No,Yes,No,Ceske Drahy_OEBB_PKP,OEBB,OEBB,depart,f,Wien Hbf,OEBB_EC_100_2025-01-04,OEBB_EC_100_2025-01-04_Austria,False,2025-01-04 17:10:00+00:00,NaN,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39574,2025-01-24,OEBB,D,domestic,Austria,No,Yes,No,OEBB,OEBB,OEBB,intermediate,f,Schladming,OEBB_D_1014_2025-01-24,OEBB_D_1014_2025-01-24_Austria,False,2025-01-24 16:48:00+00:00,180.0,False,False,False,False,False,False,False
39575,2025-01-24,OEBB,D,domestic,Austria,No,Yes,No,OEBB,OEBB,OEBB,intermediate,f,Radstadt,OEBB_D_1014_2025-01-24,OEBB_D_1014_2025-01-24_Austria,False,2025-01-24 17:08:00+00:00,120.0,False,False,False,False,False,False,False
39576,2025-01-24,OEBB,D,domestic,Austria,No,Yes,No,OEBB,OEBB,OEBB,intermediate,f,Altenmarkt im Pongau,OEBB_D_1014_2025-01-24,OEBB_D_1014_2025-01-24_Austria,False,2025-01-24 17:14:00+00:00,60.0,False,False,False,False,False,False,False
39577,2025-01-24,OEBB,D,domestic,Austria,No,Yes,No,OEBB,OEBB,OEBB,intermediate,f,Eben im Pongau,OEBB_D_1014_2025-01-24,OEBB_D_1014_2025-01-24_Austria,False,2025-01-24 17:20:00+00:00,0.0,False,False,False,False,False,False,False


In [24]:
# long distance trains only (LD == Yes), joint services included
oebb = data_chuuchuu[(data_chuuchuu["operator_for_punctuality"] == "OEBB") & (data_chuuchuu["LD"] == "Yes")]

# STOP LEVEL: every scheduled arrival in Austria (a train's departure station has no arrival, so
# it's left out). Joint services are covered: on LD trains, every stop in Austria is attributed to
# OEBB in operator_for_punctuality, whoever runs the train (Step 6 of Chuuchuu_pipeline).
# Cancellations are not counted: only stops served, with an arrival delay measured
oebb_measured = oebb[
    (oebb["country"] == "Austria")
    & (oebb["depart_terminus"] != "depart")
    & (oebb["arrivalCancelled_resolved"] == "f")
    & oebb["arrivalDelay"].notna()
]

# punctual = delay of no more than 5 min 29 s, so delayed when arrivalDelay > 329 s
# (computed here directly: the delayed_Xmin columns only cover whole minutes)
OEBB_THRESHOLD_S = 6 * 60
oebb_late = oebb_measured["arrivalDelay"] > OEBB_THRESHOLD_S

# 5 min variants: punctual = delay of no more than 5 min, so delayed when arrivalDelay > 300 s
OEBB_5MIN_S = 5 * 60
oebb_late_5min = oebb_measured["arrivalDelay"] > OEBB_5MIN_S

# TERMINUS LEVEL: one row per train, its closing stop in Austria = the terminus, or for a train that
# leaves Austria the last stop before the border (the border stop is the terminus of the Austrian
# segment). Cancellation status is read on the train's Austrian stops: partial = at least one stop
# cancelled, full = every stop cancelled
oebb_austria = oebb[oebb["country"] == "Austria"]
oebb_terminus = oebb_austria.sort_values("sort_time").groupby("journey_id", observed=True).tail(1)
oebb_terminus = oebb_terminus[oebb_terminus["depart_terminus"] != "depart"]  # departure-only segments: no arrival
share_stops_cancelled = (oebb_austria["arrivalCancelled_resolved"] == "t").groupby(oebb_austria["journey_id"], observed=True).mean()
cancelled_journeys = set(share_stops_cancelled.index[share_stops_cancelled > 0])  # full + partial
fully_cancelled_journeys = set(share_stops_cancelled.index[share_stops_cancelled == 1])

oebb_terminus = oebb_terminus.assign(
    train_cancelled=oebb_terminus["journey_id"].isin(cancelled_journeys),
    train_fully_cancelled=oebb_terminus["journey_id"].isin(fully_cancelled_journeys),
    terminus_cancelled=oebb_terminus["arrivalCancelled_resolved"] == "t",
)
# a partially cancelled train counts as cancelled only if it does not reach its terminus
oebb_terminus["not_reaching_terminus"] = oebb_terminus["train_fully_cancelled"] | (
    oebb_terminus["train_cancelled"] & oebb_terminus["terminus_cancelled"]
)

# served terminus with a measured delay (cancellations not counted)
oebb_terminus_served = oebb_terminus[~oebb_terminus["terminus_cancelled"] & oebb_terminus["arrivalDelay"].notna()]
# kept when it can be judged: train cancelled, or arrival delay measured at the terminus
t = oebb_terminus[oebb_terminus["train_cancelled"] | oebb_terminus["arrivalDelay"].notna()]
t_late = (t["arrivalDelay"] > OEBB_5MIN_S) & ~t["terminus_cancelled"]  # delayed at a served terminus

# day trains with a resolved journey_type only (for the last two rows)
keep = (t["night_train"] != "Yes") & (t["journey_type"] != "unknown")

# (threshold, measured_at, cancellation rule, rows measured, late mask)
OEBB_CASES = [
    ("6 min", "all arrivals in Austria", "not counted", oebb_measured, oebb_late),
    ("5 min", "all arrivals in Austria", "not counted", oebb_measured, oebb_late_5min),
    ("5 min", "terminus / last stop before border", "not counted", oebb_terminus_served,
        oebb_terminus_served["arrivalDelay"] > OEBB_5MIN_S),
    ("5 min", "terminus / last stop before border", "train cancelled (full + partial)", t,
        t_late | t["train_cancelled"]),
    ("5 min", "terminus / last stop before border", "full + partial cancelled; partial only if it does not reach its terminus", t,
        t_late | t["not_reaching_terminus"]),
    # same two rows, without night trains and without journeys of unknown journey_type
    ("5 min", "terminus / last stop before border, no night trains, journey_type known", "train cancelled (full + partial)", t[keep],
        (t_late | t["train_cancelled"])[keep]),
    ("5 min", "terminus / last stop before border, no night trains, journey_type known", "full + partial cancelled; partial only if it does not reach its terminus", t[keep],
        (t_late | t["not_reaching_terminus"])[keep]),
]

oebb_results = pd.DataFrame([{
    "services": "OEBB long distance services in Austria (joint services included)",
    "threshold": threshold,
    "measured_at": measured_at,
    "cancellation_counted_as_delayed": rule,
    "n_measured": len(rows),
    "n_delayed": late.sum(),  # delayed (+ cancelled where counted)
    "punctuality_chuuchuu_%": round((1 - late.mean()) * 100, 1),
} for threshold, measured_at, rule, rows, late in OEBB_CASES])

display(oebb_results)

,services,threshold,measured_at,cancellation_counted_as_delayed,n_measured,n_delayed,punctuality_chuuchuu_%
0,OEBB long distance services in Austria (joint ...,6 min,all arrivals in Austria,not counted,1015505,176452,82.6
1,OEBB long distance services in Austria (joint ...,5 min,all arrivals in Austria,not counted,1015505,199844,80.3
2,OEBB long distance services in Austria (joint ...,5 min,terminus / last stop before border,not counted,131080,30562,76.7
3,OEBB long distance services in Austria (joint ...,5 min,terminus / last stop before border,train cancelled (full + partial),133158,33569,74.8
4,OEBB long distance services in Austria (joint ...,5 min,terminus / last stop before border,full + partial cancelled; partial only if it d...,133158,32580,75.5
5,OEBB long distance services in Austria (joint ...,5 min,"terminus / last stop before border, no night t...",train cancelled (full + partial),121179,30272,75.0
6,OEBB long distance services in Austria (joint ...,5 min,"terminus / last stop before border, no night t...",full + partial cancelled; partial only if it d...,121179,29284,75.8


# Ouigo France
ART publish data for Ouigo punctuality, but only for HSR services
They consider train punctuality at terminus. A train is punctual if arriving less than 5 min late


In [25]:
ouigo = data_chuuchuu[data_chuuchuu["operator_for_punctuality"] == "Ouigo"]

# HSR services only (routeType OUIGO, all in France): Ouigo Train Classic (OGO / OTC) is left out
ouigo_hsr = ouigo[ouigo["HSR_service"] == "Yes"]

# train cancellation: partial = at least one stop cancelled, full = every stop cancelled
share_stops_cancelled = (ouigo_hsr["arrivalCancelled_resolved"] == "t").groupby(ouigo_hsr["journey_id"]).mean()
ouigo_cancelled_journeys = set(share_stops_cancelled.index[share_stops_cancelled > 0])
ouigo_fully_cancelled_journeys = set(share_stops_cancelled.index[share_stops_cancelled == 1])

# TRAIN LEVEL: one row per train, its terminus. is_cancelled = the train is fully or partially cancelled
ouigo_terminus = ouigo_hsr[ouigo_hsr["depart_terminus"] == "terminus"]
ouigo_terminus = ouigo_terminus.assign(
    is_cancelled=ouigo_terminus["journey_id"].isin(ouigo_cancelled_journeys),
    train_fully_cancelled=ouigo_terminus["journey_id"].isin(ouigo_fully_cancelled_journeys),
)

# punctual = arriving LESS than 5 min late, so a train exactly 5:00 late is delayed (arrivalDelay >= 300 s).
# delayed_5min (> 300 s) would count it as on time, so it's computed here directly.
# A NaN delay or a cancelled terminus is never delayed here (cancellation is handled by is_cancelled)
OUIGO_THRESHOLD_S = 5 * 60
ouigo_terminus = ouigo_terminus.assign(
    is_delayed=(ouigo_terminus["arrivalCancelled_resolved"] == "f")
               & (ouigo_terminus["arrivalDelay"] >= OUIGO_THRESHOLD_S),
)

# cancellations not counted (ART): the trains that reached their terminus, arrival delay measured.
# A partially cancelled train whose terminus was served stays in, judged on its delay
ouigo_served = ouigo_terminus[
    (ouigo_terminus["arrivalCancelled_resolved"] == "f") & ouigo_terminus["arrivalDelay"].notna()
]
# cancellations counted as delayed: the served trains above + every cancelled train (full or partial,
# even without a delay measured). One row per train, so a train both delayed and partially cancelled
# is counted once in n_delayed (delayed OR cancelled)
ouigo_with_cancelled = ouigo_terminus[ouigo_terminus["is_cancelled"] | ouigo_terminus.index.isin(ouigo_served.index)]

ouigo_cases = [
    ("not counted", ouigo_served, False),
    ("train cancelled (full + partial)", ouigo_with_cancelled, True),
]

ouigo_results = []
for cancellation_rule, rows, count_cancelled in ouigo_cases:
    late = rows["is_delayed"] | rows["is_cancelled"] if count_cancelled else rows["is_delayed"]
    ouigo_results.append({
        "services": "Ouigo HSR (France)",
        "threshold": "< 5 min",
        "measured_at": "terminus",
        "cancellation_counted_as_delayed": cancellation_rule,
        "n_measured": len(rows),  # trains
        # cancelled trains counted as delayed (0 where cancellations are left out)
        "n_cancelled_partial": (rows["is_cancelled"] & ~rows["train_fully_cancelled"]).sum() if count_cancelled else 0,
        "n_cancelled_full": (rows["is_cancelled"] & rows["train_fully_cancelled"]).sum() if count_cancelled else 0,
        "n_delayed_only": (rows["is_delayed"] & ~(rows["is_cancelled"] & count_cancelled)).sum(),  # late, not cancelled
        "n_delayed": late.sum(),  # delayed + cancelled, each train once
        "punctuality_chuuchuu_%": round((1 - late.mean()) * 100, 1),
    })
ouigo_results = pd.DataFrame(ouigo_results)

display(ouigo_results)

,services,threshold,measured_at,cancellation_counted_as_delayed,n_measured,n_cancelled_partial,n_cancelled_full,n_delayed_only,n_delayed,punctuality_chuuchuu_%
0,Ouigo HSR (France),< 5 min,terminus,not counted,16818,0,0,3555,3555,78.9
1,Ouigo HSR (France),< 5 min,terminus,train cancelled (full + partial),16953,172,87,3480,3739,77.9


# SBB
SBB official punctuality:
* Punctuality is taken into account at every station or stop, not only at the terminus.
* A train cancelled at a stop is not considered punctual for the stop it was due to serve.
* International trains are included, for the part of the journey within Switzerland.

Thresholds used here: all trains (long distance + regional) at 3 and 5 min of the scheduled arrival time; long distance trains at 3 and 15 min.
A stop is late from the threshold on (delay >= X min), or if it is cancelled.

In [26]:
# TEMPORARY: SBB is computed from its own export (Switzerland + the foreign part of the journeys SBB runs on)
# instead of the merged file. All stops are loaded: the train-level definition below needs every stop of a journey.
sbb_data = pq.read_table(
    output_path("sbb_2025"),
    columns=CATEGORY_COLUMNS + OTHER_COLUMNS,
    read_dictionary=CATEGORY_COLUMNS,
).to_pandas()

print(f"{len(sbb_data)} stops")
print(sbb_data["operator_for_punctuality"].value_counts())

6135371 stops
operator_for_punctuality
SBB                              3186210
Other CH operators               1399105
Panorama Express                  457648
DB                                269451
OEBB                              101456
Trenitalia                         41622
SNCF                               35633
MAV                                 7131
Flixtrain                           6965
SZ                                  4163
Ceske Drahy                         2902
ZSSK                                1588
NS                                  1165
HZ                                   739
PKP                                  260
SJ                                   112
Matterhorn Gotthard Bahn (fo)         38
SNCB                                  36
DSB                                   26
Zentralbahn                            3
CFL                                    2
Name: count, dtype: int64


In [27]:
# every stop of a train SBB runs, within Switzerland (international trains: the Swiss part only).
# A train's departure station has no arrival to measure: left out. A cancelled stop counts as late.
def measured_stops(rows):
    rows = rows[rows["depart_terminus"] != "depart"]
    rows = rows.assign(is_cancelled=rows["arrivalCancelled_resolved"] == "t")
    return rows[rows["is_cancelled"] | rows["arrivalDelay"].notna()]

swiss = sbb_data[sbb_data["country"] == "Switzerland"]
sbb_stops = measured_stops(swiss[swiss["operator_for_punctuality"] == "SBB"])

# regional proxy: the SBB trains that are not long distance + every R / S / RE-type train of the SBB feed run by
# "Other CH operators" (SBB's own regional trains can't be told apart from the other Swiss railways' in the data)
regional_stops = measured_stops(swiss[
    (swiss["operator_for_punctuality"] == "Other CH operators")
    | ((swiss["operator_for_punctuality"] == "SBB") & (swiss["LD"] != "Yes"))
])

# scope -> stops measured, and the thresholds (min) it is measured at
sbb_scopes = {
    "SBB all trains (Switzerland)": (sbb_stops, [3, 5]),
    "SBB long distance (Switzerland)": (sbb_stops[sbb_stops["LD"] == "Yes"], [3, 15]),
    "Regional (Switzerland), SBB + other Swiss operators": (regional_stops, [3, 5]),
}

sbb_results = []
for services, (rows, thresholds) in sbb_scopes.items():
    for threshold in thresholds:
        sbb_delayed = ~rows["is_cancelled"] & (rows["arrivalDelay"] >= threshold * 60)
        sbb_late = sbb_delayed | rows["is_cancelled"]
        sbb_results.append({
            "services": services,
            "threshold": f"< {threshold} min",
            "measured_at": "all stops",
            "cancellation_counted_as_delayed": "stop cancelled (full + partial)",
            "n_measured": len(rows),  # stops
            "n_cancelled": rows["is_cancelled"].sum(),
            "n_delayed_only": sbb_delayed.sum(),  # late, not cancelled
            "n_delayed": sbb_late.sum(),  # delayed + cancelled
            "punctuality_chuuchuu_%": round((1 - sbb_late.mean()) * 100, 1),
        })
sbb_results = pd.DataFrame(sbb_results)

display(sbb_results)

,services,threshold,measured_at,cancellation_counted_as_delayed,n_measured,n_cancelled,n_delayed_only,n_delayed,punctuality_chuuchuu_%
0,SBB all trains (Switzerland),< 3 min,all stops,stop cancelled (full + partial),2733840,96529,198171,294700,89.2
1,SBB all trains (Switzerland),< 5 min,all stops,stop cancelled (full + partial),2733840,96529,94591,191120,93.0
2,SBB long distance (Switzerland),< 3 min,all stops,stop cancelled (full + partial),2730214,96472,197991,294463,89.2
3,SBB long distance (Switzerland),< 15 min,all stops,stop cancelled (full + partial),2730214,96472,27565,124037,95.5
4,"Regional (Switzerland), SBB + other Swiss oper...",< 3 min,all stops,stop cancelled (full + partial),1226175,20134,62534,82668,93.3
5,"Regional (Switzerland), SBB + other Swiss oper...",< 5 min,all stops,stop cancelled (full + partial),1226175,20134,22684,42818,96.5


# SNCB
SNCB counts train as punctual with a 6 min threshold. Punctuality is calculated at the train level, measured at the terminus, but for trains passing through bruxelles, also at the first Bxl stop. 
For code simplification, we will calculate for terminus + all stops in bxl. 

Cancellations does not seem to be taken into account

SNCB calculate punctuality for their whole services. We will also calculate IC punctuality only (separately) to compare with infrabel data

In [28]:
sncb = data_chuuchuu[(data_chuuchuu["operator_for_punctuality"] == "SNCB") & (data_chuuchuu["country"] == "Belgium")]

# stations of the Brussels-Capital Region, as spelled in the data (several spellings per station).
# Brussels Airport - Zaventem is in Flanders, so it's left out
BXL_STATIONS = {
    "Brussels-South/Brussels-Midi", "Bruxelles Midi", "Brussel-Zuid",
    "Brussels-North", "Bruxelles-Nord", "Brussel-Noord",
    "Brussels-Central", "Brussel-Centraal",
    "Brussels-Luxemburg/Brussels-Luxembourg", "Brussels-Schuman", "Brussels-West", "Brussels-Congres",
    "Brussels-Chapelle/Brussels-Kapellekerk", "Schaarbeek/Schaerbeek", "Etterbeek", "Jette", "Bockstael",
    "Meiser", "Evere", "Bordet", "Sint-Agatha-Berchem/Berchem-Sainte-Agathe", "Sint-Job/Saint-Job",
    "Boondaal/Boondael", "Haren", "Haren-Sud/Haren-Zuid", "Vorst-Zuid/Forest-Midi", "Vorst-Oost/Forest-Est",
    "Ukkel-Kalevoet/Uccle-Calevoet", "Ukkel-Stalle/Uccle-Stalle", "Delta", "Merode", "Moensberg", "Simonis",
    "Anderlecht", "Watermaal/Watermael", "Arcaden", "Mouterij", "Thurn en Taxis",
}

# measuring points: the train's last stop in Belgium (terminus, or last stop before the border for
# trains going abroad) + every stop in Brussels (simplification of SNCB's "first Brussels stop").
# A stop that is both (a train ending in Brussels) is one row, so it's measured once.
# A train's departure station has no arrival, so it's left out (even in Brussels)
is_measuring_point = (
    sncb["depart_terminus"].isin(["terminus", "last stop crossborder"])
    | sncb["stopName"].isin(BXL_STATIONS)
)
# cancellations not counted: only stops served, with an arrival delay measured
sncb_measured = sncb[
    is_measuring_point
    & (sncb["depart_terminus"] != "depart")
    & (sncb["arrivalCancelled_resolved"] == "f")
    & sncb["arrivalDelay"].notna()
]

# punctual = LESS than 6 min late, so a stop exactly 6:00 late is delayed (arrivalDelay >= 360 s).
# delayed_6min (> 360 s) would count it as on time, so it's computed here directly
SNCB_THRESHOLD_S = 6 * 60

# services -> stops measured. IC only: to compare with Infrabel's data
sncb_cases = {
    "all SNCB services (Belgium)": sncb_measured,
    "SNCB IC only (Belgium)": sncb_measured[sncb_measured["routeType"] == "IC"],
}

sncb_results = []
for services, rows in sncb_cases.items():
    sncb_late = rows["arrivalDelay"] >= SNCB_THRESHOLD_S
    sncb_results.append({
        "services": services,
        "threshold": "< 6 min",
        "measured_at": "terminus + all Brussels stops",
        "cancellation_counted_as_delayed": "not counted",
        "n_measured": len(rows),  # stops
        "n_measured_terminus": rows["depart_terminus"].isin(["terminus", "last stop crossborder"]).sum(),
        "n_measured_bxl": rows["stopName"].isin(BXL_STATIONS).sum(),  # Brussels termini counted in both
        "n_delayed": sncb_late.sum(),
        "punctuality_chuuchuu_%": round((1 - sncb_late.mean()) * 100, 1),
    })
sncb_results = pd.DataFrame(sncb_results)

display(sncb_results)

,services,threshold,measured_at,cancellation_counted_as_delayed,n_measured,n_measured_terminus,n_measured_bxl,n_delayed,punctuality_chuuchuu_%
0,all SNCB services (Belgium),< 6 min,terminus + all Brussels stops,not counted,2512148,984468,1586514,236451,90.6
1,SNCB IC only (Belgium),< 6 min,terminus + all Brussels stops,not counted,930478,357721,599664,104298,88.8


In [29]:
# SNCF TGV INOUI and Intercités: same method as Ouigo (ART: train punctuality at terminus, punctual if < 5 min late)
# Only the SNCF part of each train: on international trains, the stops abroad are attributed to the
# incumbent of that country (SNCB, CFL, DB), so a train ending in Belgium / Luxembourg / Germany has
# no SNCF terminus and is left out. Trains ending in Italy / Spain stay (SNCF runs them there itself)
sncf = data_chuuchuu[data_chuuchuu["operator_for_punctuality"] == "SNCF"]

# from October to early December the source uses short route types: OUI for TGV INOUI, IC / ICN for
# Intercités / Intercités de nuit. "IC" is only Intercités for agency FR (NMBS "IC" = SNCB trains to Lille)
sncf_services = {
    "SNCF TGV INOUI": sncf[sncf["routeType"].isin(["TGV INOUI", "OUI"])],
    "SNCF Intercités (day + night)": sncf[
        sncf["routeType"].isin(["INTERCITES", "INTERCITES DE NUIT", "ICN"])
        | ((sncf["routeType"] == "IC") & (sncf["agency"] == "FR"))
    ],
}

# punctual = arriving LESS than 5 min late, so a train exactly 5:00 late is delayed (arrivalDelay >= 300 s).
SNCF_THRESHOLD_S = 5 * 60

sncf_results = []
for services, stops in sncf_services.items():
    # train cancellation: partial = at least one stop cancelled, full = every stop cancelled
    share_stops_cancelled = (stops["arrivalCancelled_resolved"] == "t").groupby(stops["journey_id"]).mean()
    cancelled_journeys = set(share_stops_cancelled.index[share_stops_cancelled > 0])
    fully_cancelled_journeys = set(share_stops_cancelled.index[share_stops_cancelled == 1])

    # TRAIN LEVEL: one row per train, its terminus. is_cancelled = the train is fully or partially cancelled.
    # A NaN delay or a cancelled terminus is never delayed here (cancellation is handled by is_cancelled)
    terminus = stops[stops["depart_terminus"] == "terminus"]
    terminus = terminus.assign(
        is_cancelled=terminus["journey_id"].isin(cancelled_journeys),
        train_fully_cancelled=terminus["journey_id"].isin(fully_cancelled_journeys),
        is_delayed=(terminus["arrivalCancelled_resolved"] == "f") & (terminus["arrivalDelay"] >= SNCF_THRESHOLD_S),
    )

    # cancellations not counted (ART): the trains that reached their terminus, arrival delay measured.
    # A partially cancelled train whose terminus was served stays in, judged on its delay
    served = terminus[(terminus["arrivalCancelled_resolved"] == "f") & terminus["arrivalDelay"].notna()]
    # cancellations counted as delayed: the served trains above + every cancelled train (full or partial,
    # even without a delay measured). One row per train, so a train both delayed and partially cancelled
    # is counted once in n_delayed (delayed OR cancelled)
    with_cancelled = terminus[terminus["is_cancelled"] | terminus.index.isin(served.index)]

    for cancellation_rule, rows, count_cancelled in [
        ("not counted", served, False),
        ("train cancelled (full + partial)", with_cancelled, True),
    ]:
        late = rows["is_delayed"] | rows["is_cancelled"] if count_cancelled else rows["is_delayed"]
        sncf_results.append({
            "services": services,
            "threshold": "< 5 min",
            "measured_at": "terminus",
            "cancellation_counted_as_delayed": cancellation_rule,
            "n_measured": len(rows),  # trains
            # cancelled trains counted as delayed (0 where cancellations are left out)
            "n_cancelled_partial": (rows["is_cancelled"] & ~rows["train_fully_cancelled"]).sum() if count_cancelled else 0,
            "n_cancelled_full": (rows["is_cancelled"] & rows["train_fully_cancelled"]).sum() if count_cancelled else 0,
            "n_delayed_only": (rows["is_delayed"] & ~(rows["is_cancelled"] & count_cancelled)).sum(),  # late, not cancelled
            "n_delayed": late.sum(),  # delayed + cancelled, each train once
            "punctuality_chuuchuu_%": round((1 - late.mean()) * 100, 1),
        })
sncf_results = pd.DataFrame(sncf_results)

display(sncf_results)

,services,threshold,measured_at,cancellation_counted_as_delayed,n_measured,n_cancelled_partial,n_cancelled_full,n_delayed_only,n_delayed,punctuality_chuuchuu_%
0,SNCF TGV INOUI,< 5 min,terminus,not counted,163279,0,0,44641,44641,72.7
1,SNCF TGV INOUI,< 5 min,terminus,train cancelled (full + partial),164062,1324,430,44003,45757,72.1
2,SNCF Intercités (day + night),< 5 min,terminus,not counted,23968,0,0,7789,7789,67.5
3,SNCF Intercités (day + night),< 5 min,terminus,train cancelled (full + partial),24394,242,333,7697,8272,66.1


# Trenitalia

In FS annual report, data fro Frecce trains, at 10 min threshold. However, it's not clear if this is measured at terminus or at all stops. Also not clear how cancellations are took into account

In [30]:
trenitalia = data_chuuchuu[data_chuuchuu["operator_for_punctuality"] == "Trenitalia"]

# Frecce route types: FR = Frecciarossa, FA = Frecciargento, FB = Frecciabianca
frecce = trenitalia[trenitalia["routeType"].isin(["FR", "FA", "FB"])]

# train cancellation: partial = at least one stop cancelled, full = every stop cancelled
share_stops_cancelled = (frecce["arrivalCancelled_resolved"] == "t").groupby(frecce["journey_id"]).mean()
frecce_cancelled_journeys = set(share_stops_cancelled.index[share_stops_cancelled > 0])
frecce_fully_cancelled_journeys = set(share_stops_cancelled.index[share_stops_cancelled == 1])

# TRAIN LEVEL: one row per train, its terminus. is_cancelled = the train is fully or partially cancelled.
# punctual = within 10 min, i.e. up to 10:00 late included, so delayed when arrivalDelay > 600 s
# (= delayed_10min: a NaN delay or a cancelled terminus is never delayed, cancellation is handled by is_cancelled)
frecce_terminus = frecce[frecce["depart_terminus"] == "terminus"]
frecce_terminus = frecce_terminus.assign(
    is_cancelled=frecce_terminus["journey_id"].isin(frecce_cancelled_journeys),
    train_fully_cancelled=frecce_terminus["journey_id"].isin(frecce_fully_cancelled_journeys),
    is_delayed=frecce_terminus["delayed_10min"],
)

# cancellations not counted: the trains that reached their terminus, arrival delay measured.
# A partially cancelled train whose terminus was served stays in, judged on its delay
frecce_served = frecce_terminus[
    (frecce_terminus["arrivalCancelled_resolved"] == "f") & frecce_terminus["arrivalDelay"].notna()
]
# cancellations counted as delayed: the served trains above + every cancelled train (full or partial,
# even without a delay measured). One row per train, so a train both delayed and partially cancelled
# is counted once in n_delayed (delayed OR cancelled)
frecce_with_cancelled = frecce_terminus[frecce_terminus["is_cancelled"] | frecce_terminus.index.isin(frecce_served.index)]

# (services, cancellation rule, trains measured, cancellations counted as delayed)
trenitalia_cases = []
for cancellation_rule, rows, count_cancelled in [
    ("not counted", frecce_served, False),
    ("train cancelled (full + partial)", frecce_with_cancelled, True),
]:
    trenitalia_cases += [
        ("Frecce (Frecciarossa + Frecciargento + Frecciabianca)", cancellation_rule, rows, count_cancelled),
        ("Frecciarossa only", cancellation_rule, rows[rows["routeType"] == "FR"], count_cancelled),
    ]

trenitalia_results = []
for services, cancellation_rule, rows, count_cancelled in trenitalia_cases:
    late = rows["is_delayed"] | rows["is_cancelled"] if count_cancelled else rows["is_delayed"]
    trenitalia_results.append({
        "services": services,
        "threshold": "<= 10 min",
        "measured_at": "terminus",
        "cancellation_counted_as_delayed": cancellation_rule,
        "n_measured": len(rows),  # trains
        # cancelled trains counted as delayed (0 where cancellations are left out)
        "n_cancelled_partial": (rows["is_cancelled"] & ~rows["train_fully_cancelled"]).sum() if count_cancelled else 0,
        "n_cancelled_full": (rows["is_cancelled"] & rows["train_fully_cancelled"]).sum() if count_cancelled else 0,
        "n_delayed_only": (rows["is_delayed"] & ~(rows["is_cancelled"] & count_cancelled)).sum(),  # late, not cancelled
        "n_delayed": late.sum(),  # delayed + cancelled, each train once
        "punctuality_chuuchuu_%": round((1 - late.mean()) * 100, 1),
    })
trenitalia_results = pd.DataFrame(trenitalia_results)

display(trenitalia_results)

,services,threshold,measured_at,cancellation_counted_as_delayed,n_measured,n_cancelled_partial,n_cancelled_full,n_delayed_only,n_delayed,punctuality_chuuchuu_%
0,Frecce (Frecciarossa + Frecciargento + Freccia...,<= 10 min,terminus,not counted,89138,0,0,21806,21806,75.5
1,Frecciarossa only,<= 10 min,terminus,not counted,79816,0,0,18940,18940,76.3
2,Frecce (Frecciarossa + Frecciargento + Freccia...,<= 10 min,terminus,train cancelled (full + partial),90343,1485,0,21606,23091,74.4
3,Frecciarossa only,<= 10 min,terminus,train cancelled (full + partial),80849,1286,0,18762,20048,75.2


# Westbahn
Westbhan definition of punctuality is :
- arriving within 5.5 min of schedule time at terminus
- a cancelled train is considered delayed
Metric measured at the train level for AT and DE routes

In [31]:
# Westbahn is scored on its whole journeys (operator_exception, Step 6 of Chuuchuu_pipeline), so its
# trains to Germany (München, Stuttgart) keep their German terminus: AT and DE routes are both covered
westbahn = data_chuuchuu[data_chuuchuu["operator_for_punctuality"] == "Westbahn"]

# train cancellation: partial = at least one stop cancelled, full = every stop cancelled
share_stops_cancelled = (westbahn["arrivalCancelled_resolved"] == "t").groupby(westbahn["journey_id"]).mean()
westbahn_cancelled_journeys = set(share_stops_cancelled.index[share_stops_cancelled > 0])
westbahn_fully_cancelled_journeys = set(share_stops_cancelled.index[share_stops_cancelled == 1])

# TRAIN LEVEL: one row per train, its terminus. is_cancelled = the train is fully or partially cancelled.
# punctual = within 5.5 min, i.e. up to 5:30 late included, so delayed when arrivalDelay > 330 s
# (computed here directly: the delayed_Xmin columns only cover whole minutes).
# A NaN delay or a cancelled terminus is never delayed here (cancellation is handled by is_cancelled)
WESTBAHN_THRESHOLD_S = 5 * 60 + 30
westbahn_terminus = westbahn[westbahn["depart_terminus"] == "terminus"]
westbahn_terminus = westbahn_terminus.assign(
    is_cancelled=westbahn_terminus["journey_id"].isin(westbahn_cancelled_journeys),
    train_fully_cancelled=westbahn_terminus["journey_id"].isin(westbahn_fully_cancelled_journeys),
    is_delayed=(westbahn_terminus["arrivalCancelled_resolved"] == "f")
               & (westbahn_terminus["arrivalDelay"] > WESTBAHN_THRESHOLD_S),
)

# the trains that reached their terminus, arrival delay measured
westbahn_served = (westbahn_terminus["arrivalCancelled_resolved"] == "f") & westbahn_terminus["arrivalDelay"].notna()

# cancellation rule -> which trains count as cancelled (= delayed). Measured: the served trains + those
# cancelled trains (even without a delay measured). One row per train, so a train both delayed and
# cancelled is counted once in n_delayed (delayed OR cancelled).
# Full cancellation only: a partially cancelled train is not counted as cancelled, it's judged on its
# delay at the terminus if it reached it, and left out otherwise
westbahn_cases = {
    "train cancelled (full + partial)": westbahn_terminus["is_cancelled"],
    "train fully cancelled only": westbahn_terminus["train_fully_cancelled"],
}

westbahn_results = []
for cancellation_rule, counted_cancelled in westbahn_cases.items():
    rows = westbahn_terminus[westbahn_served | counted_cancelled]
    cancelled = counted_cancelled[rows.index]
    late = rows["is_delayed"] | cancelled
    westbahn_results.append({
        "services": "Westbahn (AT + DE routes)",
        "threshold": "<= 5 min 30 s",
        "measured_at": "terminus",
        "cancellation_counted_as_delayed": cancellation_rule,
        "n_measured": len(rows),  # trains
        # cancelled trains counted as delayed (partial: 0 where only full cancellation is counted)
        "n_cancelled_partial": (cancelled & ~rows["train_fully_cancelled"]).sum(),
        "n_cancelled_full": (cancelled & rows["train_fully_cancelled"]).sum(),
        "n_delayed_only": (rows["is_delayed"] & ~cancelled).sum(),  # late, not counted as cancelled
        "n_delayed": late.sum(),  # delayed + cancelled, each train once
        "punctuality_chuuchuu_%": round((1 - late.mean()) * 100, 1),
    })
westbahn_results = pd.DataFrame(westbahn_results)

display(westbahn_results)

,services,threshold,measured_at,cancellation_counted_as_delayed,n_measured,n_cancelled_partial,n_cancelled_full,n_delayed_only,n_delayed,punctuality_chuuchuu_%
0,Westbahn (AT + DE routes),<= 5 min 30 s,terminus,train cancelled (full + partial),5826,130,2,1749,1881,67.7
1,Westbahn (AT + DE routes),<= 5 min 30 s,terminus,train fully cancelled only,5819,0,2,1828,1830,68.6


# SNCF open data

La régularité TGV tient compte des différentes durées de trajet des clients (aussi appelée composite).

Un train est considéré à l'heure si son retard au terminus est inférieur à 5min pour un parcours inférieur à 1h30
Un train est considéré à l'heure si son retard au terminus est inférieur à 10min pour un parcours entre 1h30 et 3h
Un train est considéré à l'heure si son retard au terminus est inférieur à 15min pour un parcours au-delà de 3h


Sur Intercités, la régularité est calculée lors de l’arrivée du train à la dernière gare de son parcours (terminus). Ce mode de calcul, qui n’intègre pas les gares intermédiaires, propose le retard cumulé sur l’ensemble d’un trajet.

L’indicateur utilisé est celui de la « régularité composite » : un train est considéré en retard s’il arrive cinq minutes après son heure prévue pour un trajet de moins de 1h30, dix minutes pour un trajet de 1h30 à 3h, et quinze minutes pour un trajet de plus de 3h.

Un train supprimé avant 16h la veille de sa circulation ne sera pas comptabilisé. En revanche, si l’annonce de sa suppression n’a pu être faite la veille avant 16h, le train sera bien comptabilisé comme un train supprimé, que cette suppression soit totale ou partielle (le train a effectué qu’une partie de son parcours). Par ailleurs, des trains Intercités peuvent effectuer une partie importante de leur trajet sous service TER. Ils sont alors comptabilisés dans les données de régularité TER.

In [32]:
# Read again from the parquet: SNCF's composite regularity needs the planned times (not in data_chuuchuu),
# and the whole journeys, stops abroad included (data_chuuchuu attributes them to SNCB / CFL / DB, and CFL
# isn't even loaded): an international train is measured at its real terminus, wherever it is.
# From October to early December the source uses short route types: OUI for TGV INOUI, IC / ICN for
# Intercités / Intercités de nuit. Only agency FR is kept: other agencies use IC / ICN for their own
# trains (NMBS IC, DB IC, Trenitalia ICN = InterCity Notte, ...)
TGV_ROUTE_TYPES = ["TGV INOUI", "OUI"]
INTERCITES_ROUTE_TYPES = ["INTERCITES", "INTERCITES DE NUIT", "ICN", "IC"]

sncf_open_data = pq.read_table(
    PARQUET_PATH,
    columns=["journey_id", "routeType", "journey_type", "depart_terminus",
             "plannedDeparture", "plannedArrival", "arrivalDelay", "arrivalCancelled_resolved"],
    filters=[("routeType", "in", TGV_ROUTE_TYPES + INTERCITES_ROUTE_TYPES), ("agency", "==", "FR")],
).to_pandas()


def composite_punctuality(stops, services, scopes):
    """SNCF composite regularity over the year: one row per train, measured at its terminus.

    A train is on time when its delay at the terminus is LESS than 5 min (journey < 1h30), 10 min
    (1h30 to 3h) or 15 min (> 3h), the journey being the planned time from departure to terminus.
    scopes: {scope name: list of journey_type kept}. Each scope gets 2 rows: cancellations
    not counted / cancelled trains (full or partial) counted as delayed.
    """
    # train cancellation (SNCF counts full and partial): at least one stop cancelled
    cancelled_journeys = set(stops.loc[stops["arrivalCancelled_resolved"] == "t", "journey_id"])

    # planned journey duration (min): planned departure at the first stop -> planned arrival at the terminus
    planned_departure = stops[stops["depart_terminus"] == "depart"].set_index("journey_id")["plannedDeparture"]
    terminus = stops[stops["depart_terminus"] == "terminus"]
    duration_min = (
        terminus["plannedArrival"] - terminus["journey_id"].map(planned_departure)
    ).dt.total_seconds() / 60

    # composite threshold (s), NaN when the duration is unknown (no departure stop)
    threshold_s = pd.Series(np.select(
        [duration_min < 90, duration_min <= 180, duration_min > 180], [5 * 60, 10 * 60, 15 * 60], np.nan,
    ), index=terminus.index)

    terminus = terminus.assign(
        duration_min=duration_min,
        is_cancelled=terminus["journey_id"].isin(cancelled_journeys),
        is_delayed=(terminus["arrivalCancelled_resolved"] == "f") & (terminus["arrivalDelay"] >= threshold_s),
    )
    # the trains that reached their terminus, arrival delay and duration known. A partially cancelled
    # train whose terminus was served is in, judged on its delay
    served = (
        (terminus["arrivalCancelled_resolved"] == "f")
        & terminus["arrivalDelay"].notna()
        & terminus["duration_min"].notna()
    )

    results = []
    for scope, journey_types in scopes.items():
        in_scope = terminus["journey_type"].isin(journey_types)
        for cancellation_rule, count_cancelled in [("not counted", False), ("train cancelled (full + partial)", True)]:
            # cancellations counted: the served trains + every cancelled train (even without a delay
            # measured). One row per train, so a train both delayed and cancelled is counted once
            measured = served | terminus["is_cancelled"] if count_cancelled else served
            rows = terminus[in_scope & measured]
            cancelled = rows["is_cancelled"] & count_cancelled
            late = rows["is_delayed"] | cancelled
            results.append({
                "services": services,
                "scope": scope,
                "threshold": "composite (< 5 / 10 / 15 min)",
                "measured_at": "terminus",
                "cancellation_counted_as_delayed": cancellation_rule,
                "n_measured": len(rows),  # trains
                "n_cancelled": cancelled.sum(),
                "n_delayed_only": (rows["is_delayed"] & ~cancelled).sum(),  # late, not cancelled
                "n_delayed": late.sum(),  # delayed + cancelled, each train once
                "punctuality_chuuchuu_%": round((1 - late.mean()) * 100, 1),
            })
    return pd.DataFrame(results)


# INSTRUCTION 1: TGV (INOUI), domestic only / domestic + international
tgv_open_data_results = composite_punctuality(
    sncf_open_data[sncf_open_data["routeType"].isin(TGV_ROUTE_TYPES)],
    services="SNCF TGV INOUI",
    scopes={"domestic": ["domestic"], "domestic + international": ["domestic", "international"]},
)

display(tgv_open_data_results)

,services,scope,threshold,measured_at,cancellation_counted_as_delayed,n_measured,n_cancelled,n_delayed_only,n_delayed,punctuality_chuuchuu_%
0,SNCF TGV INOUI,domestic,composite (< 5 / 10 / 15 min),terminus,not counted,155613,0,22444,22444,85.6
1,SNCF TGV INOUI,domestic,composite (< 5 / 10 / 15 min),terminus,train cancelled (full + partial),156435,1668,21978,23646,84.9
2,SNCF TGV INOUI,domestic + international,composite (< 5 / 10 / 15 min),terminus,not counted,168173,0,25023,25023,85.1
3,SNCF TGV INOUI,domestic + international,composite (< 5 / 10 / 15 min),terminus,train cancelled (full + partial),169133,1913,24501,26414,84.4


In [33]:
# INSTRUCTION 2: Intercités (day + night), same composite regularity at the terminus.
# Intercités are all domestic in the data, so a single scope.
# Not reproducible with our data: trains cancelled before 4pm the day before (not counted by SNCF), and
# Intercités running a large part of their journey as TER (counted in TER by SNCF)
intercites_open_data_results = composite_punctuality(
    sncf_open_data[sncf_open_data["routeType"].isin(INTERCITES_ROUTE_TYPES)],
    services="SNCF Intercités (day + night)",
    scopes={"domestic": ["domestic"]},
)

display(intercites_open_data_results)

,services,scope,threshold,measured_at,cancellation_counted_as_delayed,n_measured,n_cancelled,n_delayed_only,n_delayed,punctuality_chuuchuu_%
0,SNCF Intercités (day + night),domestic,composite (< 5 / 10 / 15 min),terminus,not counted,23926,0,4147,4147,82.7
1,SNCF Intercités (day + night),domestic,composite (< 5 / 10 / 15 min),terminus,train cancelled (full + partial),24361,575,4074,4649,80.9


# Trenitalia France (Paris - Lyon)

In [34]:
tif = data_chuuchuu[data_chuuchuu["operator_for_punctuality"] == "Trenitalia France"]

# Paris <-> Lyon trains: departure in Paris and terminus in Lyon, or the reverse. Trains only calling at
# Lyon on the way (Paris - Marseille, Paris - Milano) are left out: they're measured at another terminus.
# Lyon stations start with "Lyon" (Part-Dieu, Perrache, St-Exupéry): "Paris-Gare-de-Lyon" doesn't
origin = tif[tif["depart_terminus"] == "depart"].set_index("journey_id")["stopName"].astype(str)
destination = tif[tif["depart_terminus"] == "terminus"].set_index("journey_id")["stopName"].astype(str)
od = pd.DataFrame({"origin": origin, "destination": destination}).dropna()
is_paris = od.apply(lambda col: col.str.startswith("Paris"))
is_lyon = od.apply(lambda col: col.str.startswith("Lyon"))
paris_lyon_journeys = set(od.index[
    (is_paris["origin"] & is_lyon["destination"]) | (is_lyon["origin"] & is_paris["destination"])
])
tif_paris_lyon = tif[tif["journey_id"].isin(paris_lyon_journeys)]

# same method as Ouigo (ART: train punctuality at terminus, punctual if < 5 min late)
# train cancellation: partial = at least one stop cancelled, full = every stop cancelled
share_stops_cancelled = (tif_paris_lyon["arrivalCancelled_resolved"] == "t").groupby(tif_paris_lyon["journey_id"]).mean()
tif_cancelled_journeys = set(share_stops_cancelled.index[share_stops_cancelled > 0])
tif_fully_cancelled_journeys = set(share_stops_cancelled.index[share_stops_cancelled == 1])

# TRAIN LEVEL: one row per train, its terminus. is_cancelled = the train is fully or partially cancelled
tif_terminus = tif_paris_lyon[tif_paris_lyon["depart_terminus"] == "terminus"]
tif_terminus = tif_terminus.assign(
    is_cancelled=tif_terminus["journey_id"].isin(tif_cancelled_journeys),
    train_fully_cancelled=tif_terminus["journey_id"].isin(tif_fully_cancelled_journeys),
)

# punctual = arriving LESS than 5 min late, so a train exactly 5:00 late is delayed (arrivalDelay >= 300 s).
# A NaN delay or a cancelled terminus is never delayed here (cancellation is handled by is_cancelled)
TIF_THRESHOLD_S = 5 * 60
tif_terminus = tif_terminus.assign(
    is_delayed=(tif_terminus["arrivalCancelled_resolved"] == "f")
               & (tif_terminus["arrivalDelay"] >= TIF_THRESHOLD_S),
)

# cancellations not counted (ART): the trains that reached their terminus, arrival delay measured.
# A partially cancelled train whose terminus was served stays in, judged on its delay
tif_served = tif_terminus[
    (tif_terminus["arrivalCancelled_resolved"] == "f") & tif_terminus["arrivalDelay"].notna()
]
# cancellations counted as delayed: the served trains above + every cancelled train (full or partial,
# even without a delay measured). One row per train, so a train both delayed and partially cancelled
# is counted once in n_delayed (delayed OR cancelled)
tif_with_cancelled = tif_terminus[tif_terminus["is_cancelled"] | tif_terminus.index.isin(tif_served.index)]

tif_cases = [
    ("not counted", tif_served, False),
    ("train cancelled (full + partial)", tif_with_cancelled, True),
]

tif_results = []
for cancellation_rule, rows, count_cancelled in tif_cases:
    late = rows["is_delayed"] | rows["is_cancelled"] if count_cancelled else rows["is_delayed"]
    tif_results.append({
        "services": "Trenitalia France Paris <-> Lyon",
        "threshold": "< 5 min",
        "measured_at": "terminus",
        "cancellation_counted_as_delayed": cancellation_rule,
        "n_measured": len(rows),  # trains
        # cancelled trains counted as delayed (0 where cancellations are left out)
        "n_cancelled_partial": (rows["is_cancelled"] & ~rows["train_fully_cancelled"]).sum() if count_cancelled else 0,
        "n_cancelled_full": (rows["is_cancelled"] & rows["train_fully_cancelled"]).sum() if count_cancelled else 0,
        "n_delayed_only": (rows["is_delayed"] & ~(rows["is_cancelled"] & count_cancelled)).sum(),  # late, not cancelled
        "n_delayed": late.sum(),  # delayed + cancelled, each train once
        "punctuality_chuuchuu_%": round((1 - late.mean()) * 100, 1),
    })
tif_results = pd.DataFrame(tif_results)

display(tif_results)

,services,threshold,measured_at,cancellation_counted_as_delayed,n_measured,n_cancelled_partial,n_cancelled_full,n_delayed_only,n_delayed,punctuality_chuuchuu_%
0,Trenitalia France Paris <-> Lyon,< 5 min,terminus,not counted,2608,0,0,441,441,83.1
1,Trenitalia France Paris <-> Lyon,< 5 min,terminus,train cancelled (full + partial),2608,0,0,441,441,83.1


# Export punctuality stats

All result tables above stacked into one table (one row per operator / services / threshold / cancellation rule), written to a single Excel sheet (values only, no formulas).
The number-of-services tables below are not exported.

In [35]:
EXPORT_PATH = "summary_stats/chuuchuu_punctuality_vs_official.xlsx"
YEAR = 2025

# DB: db_results comes from the DB cell above (same columns as the other tables)

# operator -> its results tables
export_tables = {
    "DB": [db_results],
    "Eurostar": [eurostar_results],
    "NS": [ns_results],
    "OEBB": [oebb_results],
    "Ouigo": [ouigo_results],
    "SBB": [sbb_results],
    "SNCB": [sncb_results],
    "SNCF": [sncf_results],
    "SNCF open data": [tgv_open_data_results, intercites_open_data_results],
    "Trenitalia": [trenitalia_results],
    "Trenitalia France": [tif_results],
    "Westbahn": [westbahn_results],
}

export = pd.concat(
    [table.assign(operator=operator) for operator, tables in export_tables.items() for table in tables],
    ignore_index=True,
)

# the tables name the threshold either as a number of minutes (threshold_min) or as text (threshold)
export["threshold"] = export["threshold"].fillna(export["threshold_min"].astype("Int64").astype("string") + " min")
# open data tables split a service by scope
export["services"] = np.where(export["scope"].notna(), export["services"] + " (" + export["scope"] + ")", export["services"])
# one cancelled count: the tables giving partial + full cancellations sum them, the others already have it
export["n_cancelled"] = export["n_cancelled"].fillna(export["n_cancelled_full_or_partial"]).fillna(
    export["n_cancelled_partial"] + export["n_cancelled_full"])

export.insert(0, "year", YEAR)
export = export[[
    "year", "operator", "services", "threshold", "measured_at", "cancellation_counted_as_delayed",
    "n_measured", "n_cancelled_partial", "n_cancelled_full", "n_cancelled", "n_delayed_only", "n_delayed",
    "n_measured_terminus", "n_measured_bxl", "punctuality_chuuchuu_%",
]]

with pd.ExcelWriter(EXPORT_PATH, engine="openpyxl") as writer:
    export.to_excel(writer, sheet_name="Chuuchuu_table", index=False)

print(f"{len(export)} rows written to {EXPORT_PATH}")
display(export)

48 rows written to summary_stats/chuuchuu_punctuality_vs_official.xlsx


,year,operator,services,threshold,measured_at,cancellation_counted_as_delayed,n_measured,n_cancelled_partial,n_cancelled_full,n_cancelled,n_delayed_only,n_delayed,n_measured_terminus,n_measured_bxl,punctuality_chuuchuu_%
0,2025,DB,DB long distance,6 min,all stops,not counted,2875612,NaN,NaN,NaN,NaN,1182335,NaN,NaN,58.9
1,2025,Eurostar,Routes Belgium,6 min,all BE stops,not counted,32037,0.0,0.0,0.0,NaN,7164,NaN,NaN,77.6
2,2025,Eurostar,London routes - opt 1 (only arriving St Pancras),5 min,terminus,train cancelled (full + partial),9162,210.0,211.0,421.0,NaN,2356,NaN,NaN,74.3
3,2025,Eurostar,London routes - opt 2 (arriving + departing St...,5 min,terminus,train cancelled (full + partial),18270,326.0,393.0,719.0,NaN,4086,NaN,NaN,77.6
4,2025,Eurostar,London routes - opt 1 (only arriving St Pancras),15 min,terminus,train cancelled (full + partial),9162,210.0,211.0,421.0,NaN,1626,NaN,NaN,82.3
5,2025,Eurostar,London routes - opt 2 (arriving + departing St...,15 min,terminus,train cancelled (full + partial),18270,326.0,393.0,719.0,NaN,2773,NaN,NaN,84.8
6,2025,Eurostar,Continental routes,15 min,terminus,train cancelled (full + partial),22198,2142.0,1233.0,3375.0,NaN,6744,NaN,NaN,69.6
7,2025,Eurostar,Continental routes - all stops of non-London r...,15 min,all stops,stop cancelled (full + partial),74473,4252.0,2401.0,6653.0,NaN,17882,NaN,NaN,76.0
8,2025,Eurostar,Continental routes - all stops except St Pancras,15 min,all stops except St Pancras,stop cancelled (full + partial),94093,4628.0,2692.0,7320.0,NaN,20444,NaN,NaN,78.3
9,2025,Eurostar,Continental routes - all stops of non-London r...,15 min,all stops,stop of a fully cancelled train,70221,0.0,2401.0,2401.0,NaN,13630,NaN,NaN,80.6


# Double checking number of services

In [36]:
official_nb_services = pd.DataFrame({
    "Year": [2025] * 8,
    "Operator": ["Eurostar", "Eurostar", "SNCF", "SNCF", "SNCF", "SNCB", "SNCB", "SNCB"],
    "Services": [
        "London routes",
        "Continental routes on infrabel network",
        "Intercités",
        "SLO Domestiques",
        "SLO Internationaux",
        "Intercity (IC)",
        "Local (L)",
        "Peak-hour (P)",
    ],
    "Geographical scope": [
        "Infrabel network",
        "Infrabel network",
        "French domestic network",
        "French domestic network",
        "French domestic network",
        "Infrabel network",
        "Infrabel network",
        "Infrabel network",
    ],
    "Nb trains per year (official)": [6069, 12329, 28861, 201053, 61896, 551981, 689839, 65853],
})

print(official_nb_services)

   Year  Operator                                Services  \
0  2025  Eurostar                           London routes   
1  2025  Eurostar  Continental routes on infrabel network   
2  2025      SNCF                              Intercités   
3  2025      SNCF                         SLO Domestiques   
4  2025      SNCF                      SLO Internationaux   
5  2025      SNCB                          Intercity (IC)   
6  2025      SNCB                               Local (L)   
7  2025      SNCB                           Peak-hour (P)   

        Geographical scope  Nb trains per year (official)  
0         Infrabel network                           6069  
1         Infrabel network                          12329  
2  French domestic network                          28861  
3  French domestic network                         201053  
4  French domestic network                          61896  
5         Infrabel network                         551981  
6         Infrabel network    

# Number of services in Chuuchuu vs official

Same breakdown as the official table above, counted in Chuuchuu (operators of the table only: Eurostar, SNCF, SNCB). One train = one `journey_id` (one train run on one day, already deduplicated across agencies), counted when it has at least one stop in the geographical scope of the official figure (Belgium for the Infrabel network, France for the French network).

How the services are built:

* **Eurostar London routes**: trains with a stop in the United Kingdom *and* a stop in Belgium. Direct London - Paris trains never touch the Infrabel network, so they are not in the official figure and are left out here.
* **Eurostar Continental routes on Infrabel**: trains with a Belgian stop and no UK stop.
* **SNCF Intercités**: routeType `INTERCITES`, `INTERCITES DE NUIT`, `ICN`, and `IC` of agency `FR`, with a stop in France. (The `IC` of agency `NMBS` seen in France are SNCB trains, left out.)
* **SNCF SLO** (open-access high-speed: TGV INOUI, Ouigo, Lyria, TGV/ICE to Germany, Switzerland, ...): the `HSR_service == "Yes"` trains of SNCF and Ouigo with a stop in France, split by `journey_type` (domestic / international). Variant without Ouigo. Eurostar and Trenitalia France are other operators, so not included.
* **SNCB IC / L / P**: routeType `IC`, `L`, `P` of agency `NMBS`, operator SNCB, with a stop in Belgium (international extensions included, as the scope is the Infrabel network). S trains, EuroCity and ICE are not in the official table.

`n_trains_chuuchuu` counts every train, `n_trains_not_fully_cancelled` removes trains whose every stop is cancelled (the official figure may be planned or run trains, we don't know).

In [37]:
op = data_chuuchuu["operator_for_punctuality"]
country = data_chuuchuu["country"]
route_type = data_chuuchuu["routeType"]
agency = data_chuuchuu["agency"]

# ---- Eurostar: route groups from the countries each train calls in
eurostar = data_chuuchuu[op == "Eurostar"]
es_uk = set(eurostar.loc[eurostar["country"] == "United Kingdom", "journey_id"])
es_be = set(eurostar.loc[eurostar["country"] == "Belgium", "journey_id"])
es_in = lambda ids: eurostar["journey_id"].isin(ids)

# ---- SNCF: Intercités and SLO (high-speed), on the French network
sncf_group = op.isin(["SNCF", "Ouigo"]) & (country == "France")
is_intercites = (
    (op == "SNCF") & (country == "France")
    & (route_type.isin(["INTERCITES", "INTERCITES DE NUIT", "ICN"]) | ((route_type == "IC") & (agency == "FR")))
)
is_slo = sncf_group & (data_chuuchuu["HSR_service"] == "Yes")
is_ouigo = op == "Ouigo"

# ---- SNCB: IC / L / P on the Belgian (Infrabel) network
sncb_be = (op == "SNCB") & (country == "Belgium") & (agency == "NMBS")

# service name -> rows (stops) of the trains of that service, within its geographical scope
services_rows = {
    # London - Paris direct trains have no Belgian stop: outside the Infrabel scope
    ("Eurostar", "London routes"): eurostar[es_in(es_uk) & es_in(es_be)],
    ("Eurostar", "Continental routes on infrabel network"): eurostar[es_in(es_be) & ~es_in(es_uk)],
    ("Eurostar", "All routes on infrabel network"): eurostar[es_in(es_be)],  # London + continental: every train with a Belgian stop
    ("SNCF", "Intercités"): data_chuuchuu[is_intercites],
    ("SNCF", "SLO Domestiques"): data_chuuchuu[is_slo & (data_chuuchuu["journey_type"] == "domestic")],
    ("SNCF", "SLO Internationaux"): data_chuuchuu[is_slo & (data_chuuchuu["journey_type"] == "international")],
    ("SNCF", "SLO journey_type unknown"): data_chuuchuu[is_slo & (data_chuuchuu["journey_type"] == "unknown")],
    ("SNCF", "SLO Domestiques (variant: without Ouigo)"): data_chuuchuu[is_slo & ~is_ouigo & (data_chuuchuu["journey_type"] == "domestic")],
    ("SNCF", "SLO Internationaux (variant: without Ouigo)"): data_chuuchuu[is_slo & ~is_ouigo & (data_chuuchuu["journey_type"] == "international")],
    ("SNCB", "Intercity (IC)"): data_chuuchuu[sncb_be & (route_type == "IC")],
    ("SNCB", "Local (L)"): data_chuuchuu[sncb_be & (route_type == "L")],
    ("SNCB", "Peak-hour (P)"): data_chuuchuu[sncb_be & (route_type == "P")],
}

chuuchuu_rows = []
for (operator, service), rows in services_rows.items():
    fully_cancelled = (rows["arrivalCancelled_resolved"] == "t").groupby(rows["journey_id"], observed=True).all()
    n_trains = fully_cancelled.size
    chuuchuu_rows.append({
        "Operator": operator,
        "Services": service,
        "n_trains_chuuchuu": n_trains,
        "n_trains_not_fully_cancelled": n_trains - fully_cancelled.sum(),
    })
chuuchuu_nb_services = pd.DataFrame(chuuchuu_rows)

# compare with the official figures (variants of a service are matched to the same official line)
official = official_nb_services.set_index(["Operator", "Services"])["Nb trains per year (official)"]
def official_key(row):
    service = row["Services"].split(" (variant")[0]
    return official.get((row["Operator"], service if "Continental" not in service else "Continental routes on infrabel network"))
chuuchuu_nb_services["Nb trains per year (official)"] = chuuchuu_nb_services.apply(official_key, axis=1)
chuuchuu_nb_services["ratio_chuuchuu_vs_official_%"] = (
    chuuchuu_nb_services["n_trains_chuuchuu"] / chuuchuu_nb_services["Nb trains per year (official)"] * 100
).round(1)

display(chuuchuu_nb_services)

,Operator,Services,n_trains_chuuchuu,n_trains_not_fully_cancelled,Nb trains per year (official),ratio_chuuchuu_vs_official_%
0,Eurostar,London routes,6848,6754,6069.0,112.8
1,Eurostar,Continental routes on infrabel network,22183,20950,12329.0,179.9
2,Eurostar,All routes on infrabel network,29031,27704,NaN,NaN
3,SNCF,Intercités,25169,24804,28861.0,87.2
4,SNCF,SLO Domestiques,173557,173076,201053.0,86.3
5,SNCF,SLO Internationaux,34130,33974,61896.0,55.1
6,SNCF,SLO journey_type unknown,1893,1883,NaN,NaN
7,SNCF,SLO Domestiques (variant: without Ouigo),156603,156209,201053.0,77.9
8,SNCF,SLO Internationaux (variant: without Ouigo),34130,33974,61896.0,55.1
9,SNCB,Intercity (IC),364542,363498,551981.0,66.0
